# धडा ११ - मॉडेल कॉन्टेक्स्ट प्रोटोकॉल (MCP)

**मॉडेल कॉन्टेक्स्ट प्रोटोकॉल (MCP)** हा एक मुक्त मानक आहे जो एजंट्सना रनटाइममध्ये डायनॅमिकली टूल्स, संसाधने, आणि डेटा स्रोत शोधण्याची व वापरण्याची संधी देतो. एजंटमध्ये टूल्स हार्डकोड करण्याऐवजी, MCP एजंट्सना बाह्य सर्व्हर्सशी कनेक्ट होण्याची परवानगी देतो जे मागणीवर क्षमता प्रदर्शित करतात.

या धड्यात, तुम्ही शिकाल:
- MCP काय आहे आणि एजंट सिस्टीम्ससाठी का महत्त्वाचे आहे
- MCP क्लायंट-सर्व्हर आर्किटेक्चर कसे कार्य करते
- MCP-शैलीच्या टूल शोध वापरणारे एजंट कसे तयार करायचे


## सेटअप

**पूर्वअटी:**
- डिप्लॉय केलेल्या मॉडेलसह Microsoft Foundry प्रकल्प
- प्रमाणीकरणासाठी `az login` चालवा


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## मॉडेल कॉन्टेक्स्ट प्रोटोकॉल (MCP) म्हणजे काय?

MCP हे एआय एजंट्ससाठी बाह्य साधने आणि डेटा स्रोत शोधण्यासाठी आणि त्यांच्याशी संवाद साधण्यासाठी एक मानक मार्ग निश्चित करतो:

- **MCP सर्व्हर**: साधने, संसाधने, आणि संकेत एका मानक प्रोटोकॉलद्वारे उपलब्ध करून देतो
- **MCP क्लायंट**: एजंटच्या रनटाइमचे भाग जे सर्व्हरशी कनेक्ट होतात आणि उपलब्ध क्षमता शोधतात
- **डायनॅमिक डिस्कव्हरी**: एजंट्सला हार्डकोडेड साधनांची गरज नाही — ते रनटाइममध्ये काय उपलब्ध आहे ते शोधतात

हे एजंट कोड बदलण्याशिवाय नवीन क्षमता जोडता येणार्या विस्तारित एजंट सिस्टम्स तयार करण्यासाठी शक्तिशाली आहे.


## MCP कसे कार्य करते

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजंट (MCP क्लायंट) MCP सर्व्हरशी कनेक्ट होतो
2. सर्व्हर उपलब्ध साधने आणि त्यांचे स्कीमा यांची यादी पाठवितो
3. एजंट नंतर त्याच्या विचार प्रक्रियेच्या दरम्यान कुठलेही आढळलेले साधन कॉल करू शकतो
4. निकाल त्याच प्रोटोकॉलद्वारे परत येतात


## MCP टूल डिस्कव्हरीचे अनुकरण करणे

खर्‍या MCP सर्व्हरसाठी एक चालणारा सर्व्हर प्रक्रिया आवश्यक असल्याने, आपण `@tool` फंक्शन्स वापरून नमुना दाखवू ज्यामध्ये MCP-संपर्कीत निवास सेवा काय प्रदान करेल याचे अनुकरण केले आहे.

उत्पादनामध्ये, ही टूल्स स्थानिकरीत्या परिभाषित करण्याऐवजी MCP सर्व्हरवरून गतिशीलपणे सापडतील.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-शैली साधनांसह एजंट तयार करणे


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## उत्पादनातील MCP

उत्पादन वातावरणात, MCP सामर्थ्यशाली नमुने सक्षम करते:

- **डायनॅमिक टूल शोध**: एजंट MCP सर्व्हरशी कनेक्ट होतात आणि रनटाइममध्ये टूल शोधतात
- **वियोग केलेली आर्किटेक्चर**: टूल पुरवठादार एजंटपासून स्वतंत्रपणे अपडेट करू शकतात
- **आंतर-संस्थात्मक शेअरिंग**: टीम्स MCP सर्व्हरद्वारे क्षमता उघडू शकतात ज्याचा कोणताही एजंट वापर करू शकतो
- **Microsoft Agent Framework समर्थन**: MAF मध्ये `mcp` इंटिग्रेशनद्वारे अंगभूत MCP क्लायंट समर्थन आहे

MAF सह खऱ्या MCP सर्व्हरचा वापर करण्यासाठी, तुम्ही `hosted_mcp_tool()` किंवा MCP क्लायंट इंटिग्रेशनद्वारे कनेक्ट करू शकता.

**अधिक जाणून घ्या:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

या धड्यामध्ये, तुम्ही शिकलात:
- **MCP** एजंट आणि टूल प्रदात्यांदरम्यान डायनामिक टूल शोधासाठी एक खुला मानक आहे
- **क्लायंट-सर्व्हर आर्किटेक्चर** एजंट्सना रनटाइम मध्ये क्षमता शोधण्याची परवानगी देते
- MCP **विस्तारित करण्यायोग्य, स्वतंत्र एजंट प्रणाली** सक्षम करते जिथे टूल कोड बदलांशिवाय जोडले जाऊ शकतात
- Microsoft Agent Framework उत्पादन वापरासाठी **बिल्ट-इन MCP समर्थन** प्रदान करते


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
